## 1. What is Naive RAG?
RAG (Retrieval-Augmented Generation) is when you give **external data** to the prompt when asking an LLM. 

You need this because the LLM doesn't know about your private or new data (knowledge cutoff), and this data is critical—no mistakes allowed. Instead of relying on the LLM's memory, we let it "read" the right documents before answering.

### Phase 1: Indexing (Offline Preparation)
This happens once before users ask questions.
1.  **Raw Data:** You start with text (PDFs, Docs, Websites).
2.  **Chunking:** You split the text into smaller segments (e.g., 512 tokens per chunk) so they fit into the model.
3.  **Embedding:** You pass each chunk into a **Transformer Encoder Model**. 
    *   *Correction:* The model uses self-attention to understand the **semantic meaning** of the whole chunk.
    *   *Result:* Each chunk becomes **ONE dense vector** (e.g., 768 numbers), not one vector per word.
4.  **Storage:** You save these vectors in a **Vector Database**.

### Phase 2: Retrieval (Online Search)
This happens every time a user asks a question.
1.  **Query Embedding:** You take the user's question and pass it through the **SAME Embedding Model** used in Phase 1.
2.  **Similarity Search:** The query vector goes into the Vector Database. It checks similarity against all stored chunk vectors.
    *   **Metric:** It uses **Cosine Similarity** (to check if they point in the same direction) OR **Dot Product with L2 Normalization** (same result, but faster by getting rid of the magnitude).
3.  **Top-K:** The system retrieves the top `k` most similar chunks (e.g., the top 3 documents) and all go with the prompt.

### Phase 3: Generation (Online Answer)
1.  **Prompt Construction:** You take the **Retrieved Chunks** + **Original Query** and combine them into a prompt template.
2.  **Output:** The LLM generates the final answer grounded in the retrieved context

## Under the Hood" Stuff

### Vector Space Geometry (The Unit Hypersphere)
When we talk about embeddings, imagine a map with 768 dimensions (not just 2D or 3D).
*   **Raw Vectors:** Can be anywhere in space. Long documents = long vectors (large magnitude).
*   **Normalized Vectors:** We force every vector to have a length of exactly **1.0**.
*   **The Result:** All your data points now sit on the surface of a **hypersphere** (a multi-dimensional sphere).
*   **Why?** This removes "document length" bias. A short paragraph and a long chapter can now be compared fairly based purely on **direction** (meaning), not size.

###  Pooling: How 512 Tokens Become 1 Vector
A transformer outputs a vector for *every token* (word piece). How do we get one vector for the whole chunk?
*   **CLS Pooling:** The model adds a special `[CLS]` token at the start. The final vector for this token represents the whole sentence. (Common in BERT).
*   **Self Attention** Each token become a vector and these vectors go the attention so the vectors changes based on the surrounding tokens.
*   **Mean Pooling:** We take the average of all token vectors. (Common in RoBERTa / SentenceTransformers).


###  Indexing Algorithms: How Search is Fast
You don't compare your query to every single vector (Brute Force). That's too slow.
*   **HNSW (Hierarchical Navigable Small World):** Builds a graph of vectors. It "hops" from neighbor to neighbor to find the closest match quickly. 
    *   *Trade-off:* Uses more RAM, but very fast search.
*   **IVF (Inverted File Index):** Groups vectors into clusters (like folders). It only searches the closest clusters.
    *   *Trade-off:* Uses less RAM, but slightly less accurate than HNSW.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load Model (Must be the SAME for docs and queries)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Indexing Phase: Text -> Vector
chunks = [
    "Apple stock price reached new highs.",
    "Apple pie recipe with cinnamon."
]

# Embedding: Each chunk becomes ONE vector (e.g., 384 numbers)
# Math: v = Model(chunk)
chunk_vectors = model.encode(chunks) 

print(f"Chunk Vector Shape: {chunk_vectors.shape}") 
# Output: (2, 384) -> 2 chunks, 384 dimensions each
# NOTE: Not one vector per word. ONE vector per chunk.

# 3. L2 Normalization (Removing Magnitude)
# Math: v_norm = v / ||v||
# This forces every vector to have a length of exactly 1.0
norms = np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
chunk_vectors_norm = chunk_vectors / norms

# 4. Query Phase: Query -> Vector
query = "What is the stock price?"
query_vector = model.encode([query])

# Normalize Query too (Must match the index)
query_norm = np.linalg.norm(query_vector, axis=1, keepdims=True)
query_vector_norm = query_vector / query_norm

# 5. Similarity Search: Dot Product
# Math: Score = q_norm · d_norm
# Because vectors are normalized, Dot Product == Cosine Similarity
scores = np.dot(query_vector_norm, chunk_vectors_norm.T)

print(f"Similarity Scores: {scores}")
# Higher score = More semantically similar

/home/shams/venvs/common_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-07 23:25:24.098655: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Chunk Vector Shape: (2, 384)
Similarity Scores: [[0.4968732  0.08909254]]


We have a problem in standard Dense RAG when searching for a specific name (like "Honda125" or a specific ID). To solve this, we need to add an exact keyword match system (Sparse Search / BM25). 

It works by first chunking the data, tokenizing it, and building an **Inverted Index** (there are no AI models here, just the exact words as they are). Each unique word acts as a key that maps to the specific chunks containing it (e.g., `"Aspirin" -> Doc4, Doc9`).

### The Hybrid Query Process
Now that we have a Hybrid RAG architecture, when we run a query, it splits into two paths: 
1. One path goes to the **Dense (Vector)** database.
2. One path goes to the **Sparse (Inverted Index/JSON)** database to return chunks based on the exact words of the query. 

Then, we merge the chunks from the Dense search with the Sparse search. The chunks that ultimately get retrieved and sent to the LLM are the ones that get the highest combined score or rank highest across both lists (using a method like Reciprocal Rank Fusion).

### How the BM25 Engine Scores Chunks
After the inverted index finds the matches, BM25 takes the query and scores the retrieved chunks based on three rules:
1. **Term Frequency (TF):** It gives a higher score if the exact word appears multiple times inside that specific chunk.
2. **Inverse Document Frequency (IDF):** It gives a higher score if the word is rare across the entire database (so common words like "the" carry no weight).
3. **Length Normalization:** It gives a lower score if the match appears in a massive chunk instead of a small, focused chunk, preventing long documents from winning just by probability.

In [1]:
# Setup: pip install sentence-transformers rank_bm25 numpy

import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import re
import time

# ==========================================
# 1. THE DATA & PREPARATION
# ==========================================
chunks = [
    "Paracetamol is commonly used to treat a mild headache and reduce fever.",
    "A severe headache might require stronger medication like Ibuprofen.",
    "Hypertension, or high blood pressure, requires chronic management.",
    "Patient ID Honda125 requires immediate attention for severe migraines."
]

# --- ADVANCED NOTE 1: Production Tokenization ---
# In production, simple whitespace splitting `text.split()` is reckless. 
# It fails on pluralization ("headaches" vs "headache") and verb tenses.
# For specialized applications like an Arabic medical chatbot, this is critical: 
# Arabic morphology means prefixes (like 'ال' or 'ب') will completely break a basic BM25 index. 
# Industry standard: Use NLP libraries (like NLTK/SpaCy for English, or Farasa/CAMeL Tools 
# for Arabic) to perform Stemming/Lemmatization and Stop-Word removal BEFORE feeding it to BM25.
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    # A real system would lemmatize here: "headaches" -> "headache"
    return text.split()

tokenized_chunks = [tokenize(chunk) for chunk in chunks]

# ==========================================
# 2. BUILDING THE ENGINES (Offline Phase)
# ==========================================

# -- ENGINE A: Dense (Vector) --
model = SentenceTransformer('all-MiniLM-L6-v2')
# We use normalize_embeddings=True so we can use fast Dot Product later
chunk_vectors = model.encode(chunks, normalize_embeddings=True)

# -- ENGINE B: Sparse (BM25) --
# rank_bm25 handles the IDF math, the k1/b hyperparameters, and the inverted index internally.
bm25_engine = BM25Okapi(tokenized_chunks)


# ==========================================
# 3. ONLINE QUERY EXECUTION
# ==========================================
query = "Honda125 severe headache"

start_time = time.time()

# --- ADVANCED NOTE 2: Asynchronous Execution ---
# In a real backend (like FastAPI on a Linux server), you NEVER run these sequentially.
# Dense search uses HNSW graphs (O(log n)), while BM25 scoring can degrade to O(n) 
# if the query contains highly common words. You must fire Engine A and Engine B 
# concurrently using `asyncio` or threading to prevent bottlenecking your latency.

# 1. Dense Search Execution
query_vector = model.encode([query], normalize_embeddings=True)
dense_scores = np.dot(query_vector, chunk_vectors.T)[0]

# We need to map scores to chunk indices and sort them
dense_results = [{"chunk_id": i, "score": float(score)} for i, score in enumerate(dense_scores)]
dense_results = sorted(dense_results, key=lambda x: x["score"], reverse=True)

# 2. Sparse Search Execution
query_tokens = tokenize(query)
bm25_scores = bm25_engine.get_scores(query_tokens)

# Map and sort
sparse_results = [{"chunk_id": i, "score": float(score)} for i, score in enumerate(bm25_scores)]
sparse_results = sorted(sparse_results, key=lambda x: x["score"], reverse=True)


# ==========================================
# 4. RECIPROCAL RANK FUSION (The Merger)
# ==========================================

# --- ADVANCED NOTE 3: The 'k' Parameter's Real Job ---
# You know k=60 is standard, but conceptually, 'k' controls the "penalty curve".
# A smaller k (like k=10) heavily rewards chunks that appear at the very top of ONE list, 
# even if they completely fail the other.
# A larger k (like k=60 to 100) flattens the curve, meaning a chunk MUST appear in the 
# Top 10 of BOTH lists to win. We use 60 to enforce strict consensus between AI and Exact Match.

def rrf(dense_list, sparse_list, k=60):
    rrf_scores = {}
    
    for rank, item in enumerate(dense_list):
        rrf_scores[item["chunk_id"]] = 1 / (k + (rank + 1))
        
    for rank, item in enumerate(sparse_list):
        chunk_id = item["chunk_id"]
        if chunk_id in rrf_scores:
            rrf_scores[chunk_id] += 1 / (k + (rank + 1))
        else:
            rrf_scores[chunk_id] = 1 / (k + (rank + 1))
            
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

final_ranking = rrf(dense_results, sparse_results)

print(f"Query executed in {time.time() - start_time:.4f} seconds\\n")
print("--- FINAL HYBRID RANKING ---")
for rank, (chunk_id, score) in enumerate(final_ranking):
    print(f"Rank {rank + 1}: [Chunk {chunk_id}] {chunks[chunk_id][:50]}... | RRF: {score:.5f}")

/home/shams/venvs/common_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-08 17:48:13.547790: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Query executed in 0.0188 seconds\n
--- FINAL HYBRID RANKING ---
Rank 1: [Chunk 3] Patient ID Honda125 requires immediate attention f... | RRF: 0.03279
Rank 2: [Chunk 1] A severe headache might require stronger medicatio... | RRF: 0.03200
Rank 3: [Chunk 0] Paracetamol is commonly used to treat a mild heada... | RRF: 0.03200
Rank 4: [Chunk 2] Hypertension, or high blood pressure, requires chr... | RRF: 0.03125
